# Week 03 — Roboflow 음료 캔 이미지 데이터 확보·라벨링 실습

목표: 이미지 데이터셋을 직접 구성하고, Roboflow에서 음료 캔 클래스를 라벨링한 뒤 Colab에서 데이터 구조를 확인합니다.

권장 클래스 예시: `coke`, `sprite`, `fanta`


## 1. Roboflow 작업 순서
1. Roboflow 로그인
2. Workspace에서 새 Project 생성
3. Project Type은 **Object Detection** 선택
4. 이미지 업로드
5. 각 음료 캔을 Bounding Box로 표시하고 클래스명 지정
6. Annotate 완료 후 Dataset Version 생성
7. 필요하면 Resize/Auto-Orient 전처리 적용
8. Export에서 **YOLOv8** 형식 선택


In [ ]:
!pip -q install roboflow ultralytics


## 2. Roboflow 데이터셋 다운로드
아래 값은 자신의 Roboflow 프로젝트 값으로 바꿉니다. API Key는 공개 저장소에 올리지 마세요.


In [ ]:
from getpass import getpass
from roboflow import Roboflow

API_KEY = getpass('Roboflow API Key: ')
WORKSPACE = 'YOUR_WORKSPACE'
PROJECT = 'drink-can-detection'
VERSION = 1

rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download('yolov8')
print(dataset.location)


## 3. 데이터셋 폴더 구조 확인


In [ ]:
import os
for root, dirs, files in os.walk(dataset.location):
    level = root.replace(dataset.location, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files[:5]:
        print(f'{indent}  {f}')
    if level > 2:
        dirs[:] = []


## 4. data.yaml 확인


In [ ]:
from pathlib import Path
yaml_path = Path(dataset.location) / 'data.yaml'
print(yaml_path.read_text(encoding='utf-8'))


## 5. 라벨 파일 확인
YOLO 라벨 한 줄 형식은 `class_id x_center y_center width height`입니다. 값은 0~1 범위로 정규화됩니다.


In [ ]:
label_files = list((Path(dataset.location)/'train'/'labels').glob('*.txt'))
print('label files:', len(label_files))
if label_files:
    print('sample:', label_files[0].name)
    print(label_files[0].read_text())


## 6. 학습 이미지 수 확인


In [ ]:
def count_images(split):
    p = Path(dataset.location)/split/'images'
    return len([f for f in p.glob('*') if f.suffix.lower() in ['.jpg','.jpeg','.png']]) if p.exists() else 0

for split in ['train','valid','test']:
    print(split, count_images(split))


## 7. 선택 실습 — YOLOv8n 간단 학습
수업 시간이 허용될 때만 실행합니다. 데이터셋 크기와 Colab GPU 상태에 따라 시간이 달라집니다.


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(data=str(yaml_path), epochs=10, imgsz=640, batch=8)


## 8. 실습 결과 기록
- 업로드 이미지 수:
- 클래스 수:
- 클래스 이름:
- Train / Valid / Test 이미지 수:
- 라벨링하면서 가장 어려웠던 점:
- 저작권을 고려하여 데이터 수집 시 지켜야 할 점:
